In [15]:
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph,START, END
from langgraph.checkpoint.memory import InMemorySaver

from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain.agents import create_agent
from langchain.tools import tool
from pydantic import BaseModel, Field
from typing import Literal

class flowState(BaseModel):
    question:str = Field(description="User's Question Here..!")
    category:Literal['coding','google_search','weather_search'] = Field(default='google_search',
                                                                        description="Question's category here..!")
    answer:str = Field(default="") 

class check_question_category(BaseModel):
    category:Literal['coding','google_search','weather_search'] = Field(default='google_search',
                                                                            description="Question's category here..!")
llm = ChatGroq(model = "openai/gpt-oss-20b")
search = GoogleSerperAPIWrapper()
google_agent = create_agent(model = llm,
                            tools = [search.run],
                            system_prompt="you are a google agent who searches anything from google")
@tool
def get_weather(city:str):
     """It provide real time weather details for any city"""
     return f"The current temperature in {city} is 23.C"

weather_agent = create_agent(model = llm,
                             tools = [get_weather],
                             system_prompt = "you are weather agent searches weather data ")
def check_category_node(state:flowState) ->flowState:
    st_llm = llm.with_structured_output(check_question_category)
    st_category = st_llm.invoke(f"Categorize the question besed on{state.question} if not found then use 'google_search' as default category")
    state.category = st_category.category

    return state

def route(state:flowState) ->Literal['coding','google_search','weather_search']:
    return state.category

def coding_node(state:flowState):
    coding_res = llm.invoke(f"you are a coding expert {state.question}")
    state.answer = coding_res.content
    return state

def google_search_node(state:flowState):
    res = google_agent.invoke({"messages":[{"role":"user", "content":state.question}]})
    state.answer = res["messages"][-1].content
    return state

def weather_node(state:flowState):
      
      res = weather_agent.invoke({"messages":[{"role":"user", "content":state.question}]})
      state.answer = res["messages"][-1].content
      return state

graph = StateGraph(flowState)
graph.add_node("check_category_node",check_category_node)
graph.add_node("coding",coding_node)
graph.add_node("google_search",google_search_node)
graph.add_node("weather_search",weather_node)

graph.add_edge(START, "check_category_node")
graph.add_conditional_edges("check_category_node", route)
graph.add_edge("coding", END)
graph.add_edge("google_search", END)
graph.add_edge("weather_search", END)
graph = graph.compile()

res = graph.invoke({"question":"Who is the CEO of google?"})
print(res)


    



{'question': 'Who is the CEO of google?', 'category': 'google_search', 'answer': 'The current CEO of Google is **Sundar\u202fPichai**. He has held the position since 2015 and also serves as the CEO of Alphabet, Google’s parent company.'}
